# R515B Soğutucu Akışkan — Model Eğitimi

Bu notebook, Doymuş ve Kızgın buhar veri setleri için **4 farklı senaryo** ile regresyon modeli eğitimi yapar:

| # | Senaryo | Açıklama |
|---|---------|----------|
| 1 | **Base** | Orijinal veri ile temel eğitim |
| 2 | **SMOGN** | SMOGN ile veri artırma sonrası eğitim |
| 3 | **STGP-EF** | SymbolicTransformer ve Evolutionary Forest Hibrit Yöntemi ile öznitelik inşası sonrası eğitim |
| 4 | **SMOGN+STGP-EF** | Önce SMOGN, sonra STGP-EF ile birleşik eğitim |

Her senaryo **11 farklı regresyon algoritması** ile değerlendirilir: RF, ET, AdaBoost, GBDT, DART, XGBoost, LightGBM, CatBoost, KNN, GP, EF.

---

## 1. Kütüphaneler ve Fonksiyonlar

In [1]:
import sys
sys.path.insert(0, 'R515B_Ozellikler')

import pandas as pd
from Functions import (
    run_saturated_analysis,
    run_superheated_analysis,
    build_results_table,
    show_best_results,
    compare_scenarios,
    target_summary,
    save_wide_results
)

print('Fonksiyonlar başarıyla yüklendi.')

c:\Users\SGUZEY\Documents\Yüksek Lisans Dersleri\Veri Mühendisliği\Data_Engineering_Project\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-17 20:59:42,402 - INFO - Loading faiss with AVX2 support.
2026-05-17 20:59:42,420 - INFO - Successfully loaded faiss with AVX2 support.


Fonksiyonlar başarıyla yüklendi.


## 2. Veri Yükleme

In [2]:
df_doymus = pd.read_csv('R515B_Ozellikler/R515B_Doymus_Ozellikleri.csv')
df_kizgin = pd.read_csv('R515B_Ozellikler/R515B_Kizgin_Buhar.csv')

print(f'Doymuş Buhar: {df_doymus.shape}')
print(f'Kızgın Buhar: {df_kizgin.shape}')
print(f'\nDoymuş Sütunlar: {list(df_doymus.columns)}')
print(f'Kızgın Sütunlar: {list(df_kizgin.columns)}')

Doymuş Buhar: (136, 8)
Kızgın Buhar: (578, 5)

Doymuş Sütunlar: ['T(girdi)', 'P(çıktı)', 'v sıvı (çıktı)', 'v buhar (çıktı)', 'h sıvı (çıktı)', 'h buhar (çıktı)', 's sıvı (çıktı)', 's buhar (çıktı)']
Kızgın Sütunlar: ['T (girdi)', 'P (girdi)', 'v (çıktı)', 'h (çıktı)', 's (çıktı)']


In [3]:
df_doymus.head()

,T(girdi),P(çıktı),v sıvı (çıktı),v buhar (çıktı),h sıvı (çıktı),h buhar (çıktı),s sıvı (çıktı),s buhar (çıktı)
0,-30,61.2210,0.0007,0.2726,2.7282,198.2000,0.0446,0.8486
1,-29,64.2350,0.0007,0.2606,3.9422,198.8900,0.0496,0.8481
2,-28,67.3670,0.0007,0.2492,5.1587,199.5800,0.0546,0.8476
3,-27,70.6200,0.0007,0.2385,6.3775,200.2700,0.0595,0.8472
4,-26,73.9960,0.0008,0.2282,7.5988,200.9500,0.0645,0.8468


In [4]:
df_kizgin.head()

,T (girdi),P (girdi),v (çıktı),h (çıktı),s (çıktı)
0,-30,50,0.3357,198.5500,0.8639
1,-25,50,0.3433,202.4500,0.8798
2,-20,50,0.3508,206.4000,0.8956
3,-15,50,0.3582,210.4000,0.9112
4,-10,50,0.3656,214.4300,0.9267


---
## 3. Doymuş Buhar — Eğitim

**Girdi:** T(girdi)  
**Çıktılar (her biri ayrı ayrı):** P, v sıvı, v buhar, h sıvı, h buhar, s sıvı, s buhar

In [5]:
doymus_results = run_saturated_analysis(df_doymus)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  DOYMUŞ BUHAR ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Target: P(çıktı)  |  Input: ['T(girdi)']
  [1/4] Base training...
  [2/4] SMOGN training...
Özel SMOGN Algoritması başlatılıyor (Custom Build)...
Özel SMOGN Başarılı! Orijinal: 95 -> Artırılmış/Dengelenmiş: 112


2026-05-17 21:08:22,304 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [3/4] STGP-EF training...


2026-05-17 21:08:56,418 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-17 21:08:56,422 - INFO - SYMBOLIC REGRESSION (STGP) - Generated Features (10 of 10)
2026-05-17 21:08:56,431 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-17 21:08:56,439 - INFO -   STGP_00: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,439 - INFO -   STGP_01: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,439 - INFO -   STGP_02: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,465 - INFO -   STGP_03: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,516 - INFO -   STGP_04: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,552 - INFO -   STGP_05: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,570 - INFO -   STGP_06: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,586 - INFO -   STGP_07: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,600 - INFO -   STGP_08: (x0 + ((x0 * 0.352) * x0))
2026-05-17 21:08:56,615 - INFO -   STGP_09: 

  [4/4] SMOGN + STGP-EF training...


...

  [3/4] STGP-EF training...


2026-05-17 22:16:32,044 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:16:32,045 - INFO - SYMBOLIC REGRESSION (STGP) - Generated Features (10 of 10)
2026-05-17 22:16:32,045 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:16:32,046 - INFO -   STGP_00: (((((x0 * x0) + (0.888 + x0)) + (0.888 + x0)) + (0.888 + (((x0 * x0) + (0.888 + (0.888 + x0))) + 0.888))) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-17 22:16:32,046 - INFO -   STGP_01: ((((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) + (0.888 + (0.888 + x0))) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-17 22:16:32,047 - INFO -   STGP_02: ((((x0 * x0) + (0.888 + x0)) + (0.888 + 0.888)) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-17 22:16:32,048 - INFO -   STGP_03: ((((x0 * x0) + (0.888 + (0.888 + x0))) + 0.

  [4/4] SMOGN + STGP-EF training...


2026-05-17 22:19:41,370 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:19:41,371 - INFO - SYMBOLIC REGRESSION (STGP) - Generated Features (10 of 10)
2026-05-17 22:19:41,371 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:19:41,371 - INFO -   STGP_00: (((((x0 * -0.133) / (-0.233 / x0)) + (0.888 + x0)) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-17 22:19:41,372 - INFO -   STGP_01: (((((x0 * -0.133) / (-0.233 / x0)) + (0.888 + x0)) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-17 22:19:41,373 - INFO -   STGP_02: ((((x0 * x0) + (0.888 + (0.888 + x0))) + 0.888) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.888 + x0)))
2026-05-17 22:19:41,374 - INFO -   STGP_03: (((((x0 * x0) + (0.888 + x0)) + 0.888) + (0.209 - -0.839)) * ((((x0 * -0.133) / (-0.233 / x0)) - (0.209 - -0.839)) * (0.

  Completed.



### 3.1 Doymuş Buhar — Sonuç Tablosu

In [6]:
df_doymus_results = build_results_table(doymus_results)
df_doymus_results

,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAPE,Test_MAPE
0,P(çıktı),Base,AdaBoost,0.9964,0.9959,0.0598,0.0653,0.0706,0.1070
1,P(çıktı),Base,CatBoost,0.9998,0.9991,0.0123,0.0300,0.0484,0.0844
2,P(çıktı),Base,DART,0.9387,0.9541,0.2475,0.2177,0.2042,0.1343
3,P(çıktı),Base,EF,1.0000,1.0000,0.0000,0.0054,0.0000,0.0077
4,P(çıktı),Base,ET,1.0000,1.0000,0.0000,0.0035,0.0000,0.0034
...,...,...,...,...,...,...,...,...,...
303,s buhar (çıktı),SMOGN+STGP-EF,GP,0.9727,0.9350,0.2045,0.1943,0.5887,0.4809
304,s buhar (çıktı),SMOGN+STGP-EF,KNN,0.9954,0.9935,0.0836,0.0613,0.1426,0.1212
305,s buhar (çıktı),SMOGN+STGP-EF,LightGBM,0.8438,0.8265,0.4892,0.3175,0.2619,0.2665
306,s buhar (çıktı),SMOGN+STGP-EF,RF,0.9992,0.9974,0.0352,0.0388,0.0369,0.0850


### 3.2 Doymuş Buhar — Senaryo Karşılaştırması

In [7]:
compare_scenarios(df_doymus_results)

Scenario                   Base  SMOGN  STGP-EF  SMOGN+STGP-EF
Target         Algorithm                                      
P(çıktı)       AdaBoost  0.9959 0.9978   0.9980         0.9939
               CatBoost  0.9991 0.9993   0.9996         0.9995
               DART      0.9541 0.9883   0.9569         0.9895
               EF        1.0000 0.9999   1.0000         1.0000
               ET        1.0000 1.0000   1.0000         1.0000
...                         ...    ...      ...            ...
v sıvı (çıktı) GP        0.9595 0.9525   0.8472         0.8915
               KNN       0.9992 0.9992   0.9987         0.9994
               LightGBM  0.9346 0.9914   0.9699         0.9925
               RF        0.9994 0.9990   0.9996         0.9990
               XGBoost   0.9966 0.9973   0.9966         0.9971

[77 rows x 4 columns]

### 3.3 Doymuş Buhar — Hedef Bazlı Detay

Her bir hedef değişken için detaylı pivot tablo;

In [8]:
target_summary(df_doymus_results, 'P(çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9959,0.9978,0.9980,0.9939
CatBoost,0.9991,0.9993,0.9996,0.9995
DART,0.9541,0.9883,0.9569,0.9895
EF,1.0000,0.9999,1.0000,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9991,0.9992,0.9993,0.9993
GP,0.9702,0.9702,0.9952,0.8904
KNN,0.9998,0.9994,0.9996,0.9995
LightGBM,0.9651,0.9962,0.9669,0.9982


In [9]:
target_summary(df_doymus_results, 'v sıvı (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9943,0.9957,0.9968,0.9931
CatBoost,0.9981,0.9980,0.9990,0.9985
DART,0.9252,0.9845,0.9386,0.9853
EF,1.0000,0.9996,1.0000,0.9998
ET,1.0000,0.9997,1.0000,0.9997
GBDT,0.9980,0.9976,0.9987,0.9978
GP,0.9595,0.9525,0.8472,0.8915
KNN,0.9992,0.9992,0.9987,0.9994
LightGBM,0.9346,0.9914,0.9699,0.9925


In [10]:
target_summary(df_doymus_results, 'v buhar (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9875,0.9878,0.9921,0.9918
CatBoost,0.9966,0.9977,0.9977,0.9979
DART,0.7834,0.9386,0.8122,0.9425
EF,0.9987,0.9991,0.9991,0.9985
ET,0.9991,0.9991,0.9992,0.9991
GBDT,0.9959,0.9966,0.9978,0.9979
GP,0.9758,0.9758,0.8517,0.9094
KNN,0.9915,0.9932,0.9912,0.9943
LightGBM,0.7845,0.9470,0.8152,0.9485


In [11]:
target_summary(df_doymus_results, 'h sıvı (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9989,0.9980,0.9988,0.9987
CatBoost,0.9993,0.9994,0.9993,0.9993
DART,0.9667,0.9889,0.9702,0.9903
EF,0.9999,1.0000,1.0000,1.0000
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9993,0.9993,0.9994,0.9994
GP,0.9955,0.9955,0.9955,0.9955
KNN,0.9997,0.9993,0.9996,0.9994
LightGBM,0.9753,0.9960,0.9787,0.9979


In [12]:
target_summary(df_doymus_results, 'h buhar (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9948,0.9937,0.9983,0.9953
CatBoost,0.9991,0.9993,0.9994,0.9994
DART,0.9639,0.9881,0.9702,0.9891
EF,0.9999,0.9999,0.9998,0.9999
ET,1.0000,0.9999,0.9999,1.0000
GBDT,0.9991,0.9989,0.9994,0.9993
GP,0.9573,0.9573,0.9573,0.9573
KNN,0.9994,0.9991,0.9992,0.9991
LightGBM,0.9709,0.9945,0.9871,0.9978


In [13]:
target_summary(df_doymus_results, 's sıvı (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9991,0.9983,0.9993,0.9968
CatBoost,0.9991,0.9994,0.9993,0.9994
DART,0.9655,0.9887,0.9696,0.9897
EF,1.0000,1.0000,1.0000,0.9999
ET,1.0000,1.0000,1.0000,1.0000
GBDT,0.9993,0.9992,0.9995,0.9993
GP,0.9998,0.9998,0.9998,0.9998
KNN,0.9996,0.9992,0.9995,0.9993
LightGBM,0.9734,0.9955,0.9779,0.9969


In [14]:
target_summary(df_doymus_results, 's buhar (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9108,0.8847,0.9686,0.9755
CatBoost,0.9829,0.9854,0.9871,0.9956
DART,0.4696,0.3690,0.9230,0.8605
EF,0.9995,0.9981,0.9996,0.9985
ET,0.9997,0.9979,0.9990,0.9990
GBDT,0.9808,0.9800,0.9901,0.9935
GP,0.3077,0.8857,0.9676,0.9350
KNN,0.9886,0.9934,0.9822,0.9935
LightGBM,0.4773,0.2950,0.9660,0.8265


---
## 4. Kızgın Buhar — Eğitim

**Girdiler:** T (girdi), P (girdi)  
**Çıktılar (her biri ayrı ayrı):** v, h, s

In [15]:
kizgin_results = run_superheated_analysis(df_kizgin)


▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓
  KIZGIN BUHAR ANALİZİ
▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

  Target: v (çıktı)  |  Input: ['T (girdi)', 'P (girdi)']
  [1/4] Base training...
  [2/4] SMOGN training...
Özel SMOGN Algoritması başlatılıyor (Custom Build)...
Özel SMOGN Başarılı! Orijinal: 404 -> Artırılmış/Dengelenmiş: 470


2026-05-17 22:25:05,378 - INFO - Hibrit Özellik İnşası (STGP-EF) Başlatılıyor...


  [3/4] STGP-EF training...


2026-05-17 22:25:13,127 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:25:13,128 - INFO - SYMBOLIC REGRESSION (STGP) - Generated Features (10 of 10)
2026-05-17 22:25:13,128 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:25:13,129 - INFO -   STGP_00: ((x1 - (x1 * (x1 * 0.398))) * ((x1 * ((x1 - -0.624) - (x1 * (x1 * 0.398)))) * ((((x1 - (x1 * (x1 * 0.398))) * ((x1 * ((x1 - -0.624) - (x1 * (x1 * 0.398)))) * (((x1 * x1) - (x1 + x1)) * x1))) - (x1 + x1)) * x1)))
2026-05-17 22:25:13,129 - INFO -   STGP_01: ((x1 - (x1 * (x1 * 0.398))) * ((x1 * ((x1 - -0.624) - (x1 * (x1 * 0.398)))) * (((x1 * (x1 - (x1 * (x1 * 0.398)))) * (-0.251 + ((x1 * x1) - (x1 + x1)))) * x1)))
2026-05-17 22:25:13,130 - INFO -   STGP_02: (((x1 * x1) - (x1 + x1)) * ((x1 * ((x1 - -0.624) - (x1 * (x1 * 0.398)))) * ((x1 * ((x1 - -0.624) - (x1 * (x1 * 0.398)))) * (((x1 * x1) - (x1 + x1)) * x1))))
2026-05-17 22:25:13,13

  [4/4] SMOGN + STGP-EF training...


...

  [3/4] STGP-EF training...


2026-05-17 22:45:45,228 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:45:45,229 - INFO - SYMBOLIC REGRESSION (STGP) - Generated Features (10 of 10)
2026-05-17 22:45:45,230 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:45:45,231 - INFO -   STGP_00: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:45:45,232 - INFO -   STGP_01: ((((x0 * 0.31) * x1) + x0) - x1)
2026-05-17 22:45:45,232 - INFO -   STGP_02: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:45:45,233 - INFO -   STGP_03: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:45:45,234 - INFO -   STGP_04: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:45:45,234 - INFO -   STGP_05: ((((x0 * 0.31) * x1) + x0) - x1)
2026-05-17 22:45:45,234 - INFO -   STGP_06: ((((x0 * 0.31) * x1) + x0) - x1)
2026-05-17 22:45:45,235 - INFO -   STGP_07: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:45:45,235 - INFO -   STGP_08: (((x1 * (x0 * 0.31)) + 

  [4/4] SMOGN + STGP-EF training...


2026-05-17 22:49:59,333 - INFO - 
──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:49:59,335 - INFO - SYMBOLIC REGRESSION (STGP) - Generated Features (10 of 10)
2026-05-17 22:49:59,337 - INFO - ──────────────────────────────────────────────────────────────────────────────
2026-05-17 22:49:59,337 - INFO -   STGP_00: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,337 - INFO -   STGP_01: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,337 - INFO -   STGP_02: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,337 - INFO -   STGP_03: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,350 - INFO -   STGP_04: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,352 - INFO -   STGP_05: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,356 - INFO -   STGP_06: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,363 - INFO -   STGP_07: (((x1 * (x0 * 0.31)) + x0) - x1)
2026-05-17 22:49:59,365 - INFO -   STGP_08: (((x1 * (x0 * 0.31)) + 

  Completed.



### 4.1 Kızgın Buhar — Sonuç Tablosu

In [16]:
df_kizgin_results = build_results_table(kizgin_results)
df_kizgin_results

,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAPE,Test_MAPE
0,v (çıktı),Base,AdaBoost,0.9928,0.9891,0.0849,0.0984,1.0603,0.2023
1,v (çıktı),Base,CatBoost,0.9999,0.9996,0.0073,0.0177,0.0532,0.0266
2,v (çıktı),Base,DART,0.8654,0.8862,0.3669,0.3181,0.3296,0.1550
3,v (çıktı),Base,EF,1.0000,0.9998,0.0001,0.0133,0.0000,0.0135
4,v (çıktı),Base,ET,1.0000,0.9998,0.0000,0.0128,0.0000,0.0133
...,...,...,...,...,...,...,...,...,...
127,s (çıktı),SMOGN+STGP-EF,GP,0.8181,0.8439,0.5154,0.3712,0.3333,0.6943
128,s (çıktı),SMOGN+STGP-EF,KNN,0.9942,0.9813,0.0921,0.1286,0.1183,0.2880
129,s (çıktı),SMOGN+STGP-EF,LightGBM,0.9996,0.9945,0.0235,0.0695,0.0355,0.3514
130,s (çıktı),SMOGN+STGP-EF,RF,0.9995,0.9930,0.0283,0.0785,0.0405,0.3872


### 4.2 Kızgın Buhar — Senaryo Karşılaştırması

In [17]:
compare_scenarios(df_kizgin_results)

Scenario              Base  SMOGN  STGP-EF  SMOGN+STGP-EF
Target    Algorithm                                      
h (çıktı) AdaBoost  0.9662 0.9671   0.9972         0.9965
          CatBoost  0.9980 0.9974   0.9989         0.9986
          DART      0.9604 0.9738   0.9750         0.9865
          EF        0.9995 0.9993   0.9990         0.9995
          ET        0.9994 0.9994   0.9994         0.9995
          GBDT      0.9977 0.9975   0.9990         0.9992
          GP        0.9553 0.9783   0.9881         0.9662
          KNN       0.9934 0.9929   0.9976         0.9981
          LightGBM  0.9729 0.9851   0.9857         0.9949
          RF        0.9973 0.9968   0.9989         0.9992
          XGBoost   0.9979 0.9979   0.9987         0.9988
s (çıktı) AdaBoost  0.8883 0.8816   0.9735         0.9748
          CatBoost  0.9957 0.9956   0.9973         0.9978
          DART      0.9382 0.9591   0.9783         0.9850
          EF        0.9982 0.9975   0.9950         0.9965
          ET        0.9972 0.9979   0.9924         0.9956
          GBDT      0.9922 0.9888   0.9969         0.9963
          GP        0.7252 0.7686   0.8439         0.8439
          KNN       0.9686 0.9659   0.9862         0.9813
          LightGBM  0.9529 0.9761   0.9884         0.9945
          RF        0.9843 0.9877   0.9890         0.9930
          XGBoost   0.9927 0.9914   0.9944         0.9950
v (çıktı) AdaBoost  0.9891 0.9808   0.9897         0.9909
          CatBoost  0.9996 0.9987   0.9997         0.9978
          DART      0.8862 0.9889   0.9709         0.9927
          EF        0.9998 0.9988   0.9929         0.9948
          ET        0.9998 0.9992   0.9972         0.9975
          GBDT      0.9994 0.9995   0.9987         0.9985
          GP        0.0405 0.3516   0.8734         0.6103
          KNN       0.7909 0.6580   0.9706         0.9977
          LightGBM  0.8781 0.9980   0.9945         0.9970
          RF        0.9994 0.9980   0.9982         0.9974
          XGBoost   0.9990 0.9988   0.9990         0.9983

### 4.3 Doymuş Buhar — Hedef Bazlı Detay

Her bir hedef değişken için detaylı pivot tablo;

In [18]:
target_summary(df_kizgin_results, 'v (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9891,0.9808,0.9897,0.9909
CatBoost,0.9996,0.9987,0.9997,0.9978
DART,0.8862,0.9889,0.9709,0.9927
EF,0.9998,0.9988,0.9929,0.9948
ET,0.9998,0.9992,0.9972,0.9975
GBDT,0.9994,0.9995,0.9987,0.9985
GP,0.0405,0.3516,0.8734,0.6103
KNN,0.7909,0.6580,0.9706,0.9977
LightGBM,0.8781,0.9980,0.9945,0.9970


In [19]:
target_summary(df_kizgin_results, 'h (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.9662,0.9671,0.9972,0.9965
CatBoost,0.9980,0.9974,0.9989,0.9986
DART,0.9604,0.9738,0.9750,0.9865
EF,0.9995,0.9993,0.9990,0.9995
ET,0.9994,0.9994,0.9994,0.9995
GBDT,0.9977,0.9975,0.9990,0.9992
GP,0.9553,0.9783,0.9881,0.9662
KNN,0.9934,0.9929,0.9976,0.9981
LightGBM,0.9729,0.9851,0.9857,0.9949


In [20]:
target_summary(df_kizgin_results, 's (çıktı)')

Scenario,Base,SMOGN,STGP-EF,SMOGN+STGP-EF
Algorithm,,,,
AdaBoost,0.8883,0.8816,0.9735,0.9748
CatBoost,0.9957,0.9956,0.9973,0.9978
DART,0.9382,0.9591,0.9783,0.9850
EF,0.9982,0.9975,0.9950,0.9965
ET,0.9972,0.9979,0.9924,0.9956
GBDT,0.9922,0.9888,0.9969,0.9963
GP,0.7252,0.7686,0.8439,0.8439
KNN,0.9686,0.9659,0.9862,0.9813
LightGBM,0.9529,0.9761,0.9884,0.9945


# En İyi Sonuçlar

Her hedef değişken için en iyi (Algoritma, Senaryo) kombinasyonunu döndürür.

In [21]:
show_best_results(df_doymus_results)

,Target,Scenario,Algorithm,Test_R2,Train_R2,Test_RMSE,Test_MAPE
0,P(çıktı),Base,ET,1.0000,1.0000,0.0035,0.0034
1,h buhar (çıktı),Base,ET,1.0000,1.0000,0.0062,0.0065
2,h sıvı (çıktı),STGP-EF,ET,1.0000,1.0000,0.0044,0.0131
3,s buhar (çıktı),Base,ET,0.9997,1.0000,0.0134,0.0325
4,s sıvı (çıktı),STGP-EF,ET,1.0000,1.0000,0.0049,0.0082
5,v buhar (çıktı),STGP-EF,ET,0.9992,1.0000,0.0299,0.0151
6,v sıvı (çıktı),STGP-EF,ET,1.0000,1.0000,0.0051,0.0112


---
## 5. Genel Karşılaştırma

In [22]:

df_doymus_results['Veri Seti'] = 'Doymuş Buhar'
df_kizgin_results['Veri Seti'] = 'Kızgın Buhar'
df_all = pd.concat([df_doymus_results, df_kizgin_results], ignore_index=True)

print(f'Toplam değerlendirme sayısı: {len(df_all)}')
df_all.head(20)

Toplam değerlendirme sayısı: 440


,Target,Scenario,Algorithm,Train_R2,Test_R2,Train_RMSE,Test_RMSE,Train_MAPE,Test_MAPE,Veri Seti
0,P(çıktı),Base,AdaBoost,0.9964,0.9959,0.0598,0.0653,0.0706,0.1070,Doymuş Buhar
1,P(çıktı),Base,CatBoost,0.9998,0.9991,0.0123,0.0300,0.0484,0.0844,Doymuş Buhar
2,P(çıktı),Base,DART,0.9387,0.9541,0.2475,0.2177,0.2042,0.1343,Doymuş Buhar
3,P(çıktı),Base,EF,1.0000,1.0000,0.0000,0.0054,0.0000,0.0077,Doymuş Buhar
4,P(çıktı),Base,ET,1.0000,1.0000,0.0000,0.0035,0.0000,0.0034,Doymuş Buhar
5,P(çıktı),Base,GBDT,1.0000,0.9991,0.0004,0.0308,0.0011,0.0752,Doymuş Buhar
6,P(çıktı),Base,GP,0.9743,0.9702,0.1603,0.1753,0.3648,0.3469,Doymuş Buhar
7,P(çıktı),Base,KNN,0.9991,0.9998,0.0304,0.0158,0.0331,0.0438,Doymuş Buhar
8,P(çıktı),Base,LightGBM,0.9461,0.9651,0.2322,0.1898,0.1671,0.1121,Doymuş Buhar
9,P(çıktı),Base,RF,0.9999,0.9997,0.0098,0.0181,0.0186,0.0399,Doymuş Buhar


In [23]:
# Senaryo bazlı ortalama Test R² skorları
df_all.groupby(['Veri Seti', 'Senaryo'])['Test_R2'].mean().unstack()

KeyError: 'Senaryo'

In [ ]:
# Algoritma bazlı ortalama Test R² skorları
df_all.groupby(['Veri Seti', 'Algoritma'])['Test_R2'].mean().unstack()

In [ ]:
# Geniş formatlı sonuç tablosu oluştur ve kaydet
df_wide = save_wide_results(df_all, 'R515B_Ozellikler/Model_Training_Results.csv')
print('Sonuçlar R515B_Ozellikler/Model_Training_Results.csv dosyasına kaydedildi.')
print(f'Format: Geniş tablo — {len(df_wide)} satır, {len(df_wide.columns)} sütun')
df_wide